In [1]:
import json

def read_json(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)
    return data

def save_json(data, filename="formatted_references.json"):
    """Save formatted data to a JSON file."""
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

In [2]:
# Sample JSON file
json_file = "reactions.VertexReaction"
# Read and process JSON data
json_data = read_json(json_file+ ".json")
save_json(json_data, json_file+ ".json")

In [3]:
# define the new TPMethod you want to add
gas_tp_method = {
    "method": {"40": "mv_pvnrt"}
}

master_other_tp_method = {
    "method": {"34": "mv_constant"}
}

reaction_tp_method = {
    "method": {"13": "dr_volume_constant"}
}

reaction_tp_method2 = {
    "method": {"7": "logk_3_term_extrap"}
}


In [4]:
def process_items(data, method, what="append", conditions=True):
    """Process item dSDref and update item dSDval based on reference mapping."""

    for item in data:
        props = item.get("properties", {})

        if conditions:
            # CONDITION 1: sm_volume must exist
            if "sm_volume" not in props:
                continue

            # CONDITION 2: aggregate_state must contain AS_CRYSTAL
            agg = props.get("aggregate_state", {})
            if "AS_CRYSTAL" not in agg.values():
                continue

        tp_methods = props.setdefault("TPMethods", [])

        # CONDITION 3: skip if any method is HKF or water EOS
        has_hkf = any(
            any(v in ("solute_hkf88_gems", "water_eos_hgk84_lvs83_gems")
                for v in method_entry.get("method", {}).values())
            for method_entry in tp_methods
        )

        if not has_hkf:
            if what == "append":
                tp_methods.append(method)

            elif what == "replace":
                if tp_methods:
                    tp_methods[0]["method"] = method["method"]
                else:
                    tp_methods.append(method)

    return data

In [5]:
processed_data = process_items(json_data, reaction_tp_method2, "replace", False)

In [6]:
# Save formatted data to a JSON file
save_json(processed_data, json_file+ "_formatted"+ ".json")